In [15]:
# Task 1: Cài đặt và nạp các thư viện cần thiết
import os
import time
import hashlib
import sqlite3
import requests
from datetime import datetime
from collections import deque
from urllib.parse import urlparse, urljoin
from bs4 import BeautifulSoup
import re

print("Import thư viện thành công!")

Import thư viện thành công!


In [16]:
# Task 2: Cấu hình Crawler cho Nature
TOPIC = "Science"
SEED_URL = "https://www.nature.com/"
ALLOWED_DOMAINS = [
    "nature.com",
    "www.nature.com"
]
MAX_DEPTH = 2
MAX_PAGES = 30
REQUEST_TIMEOUT = 10
CRAWL_DELAY = 1.0
DB_PATH = "data/nature_crawler.db"
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

print("=== CRAWLER CONFIGURATION ===")
print(f"Topic          : {TOPIC}")
print(f"Seed URL       : {SEED_URL}")
print(f"Allowed Domains: {ALLOWED_DOMAINS}")
print(f"Max Depth      : {MAX_DEPTH} | Max Pages: {MAX_PAGES}")
print(f"Crawl Delay    : {CRAWL_DELAY}s")

=== CRAWLER CONFIGURATION ===
Topic          : Science
Seed URL       : https://www.nature.com/
Allowed Domains: ['nature.com', 'www.nature.com']
Max Depth      : 2 | Max Pages: 30
Crawl Delay    : 1.0s


In [17]:
# Task 3: URL Frontier (BFS & visited set)
class URLFrontier:
    def __init__(self, max_depth=3):
        self.queue = deque()
        self.visited = set()
        self.max_depth = max_depth
        self.discovered_count = 0
        self.skipped_count = 0

    def add_url(self, url, depth):
        self.discovered_count += 1
        if depth > self.max_depth or url in self.visited:
            self.skipped_count += 1
            return False
        self.visited.add(url)
        self.queue.append((url, depth))
        return True

    def get_next(self):
        return self.queue.popleft() if self.queue else None

    def is_empty(self):
        return len(self.queue) == 0

frontier = URLFrontier(max_depth=MAX_DEPTH)
print("Đã khởi tạo URL Frontier.")

Đã khởi tạo URL Frontier.


In [18]:
# Task 4: Duplicate Detection (SHA-256)
class DuplicateDetector:
    def __init__(self):
        self.fingerprints = set()

    def is_duplicate(self, text_content):
        if not text_content:
            return True
        fp = hashlib.sha256(text_content.strip().encode("utf-8")).hexdigest()
        if fp in self.fingerprints:
            return True
        self.fingerprints.add(fp)
        return False

detector = DuplicateDetector()
print("Đã khởi tạo Duplicate Detector.")

Đã khởi tạo Duplicate Detector.


In [19]:
# Task 5: Cơ sở dữ liệu SQLite (CÙNG CẤU TRÚC BẢNG)
class Database:
    def __init__(self, db_path):
        db_dir = os.path.dirname(db_path)
        if db_dir:
            os.makedirs(db_dir, exist_ok=True)

        self.conn = sqlite3.connect(db_path)
        self.create_tables()

    def create_tables(self):
        cursor = self.conn.cursor()

        # Bảng pages: Giữ nguyên 100% đúng cấu trúc cũ
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS pages (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                url TEXT UNIQUE,
                domain TEXT,
                title TEXT,
                abstract TEXT,
                authors TEXT,
                content TEXT,
                is_article INTEGER DEFAULT 0,
                depth INTEGER,
                status_code INTEGER,
                crawled_at TEXT
            )
        ''')

        # Bảng links: Giữ nguyên 100% đúng cấu trúc cũ
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS links (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                source_url TEXT,
                target_url TEXT
            )
        ''')

        cursor.execute('CREATE INDEX IF NOT EXISTS idx_pages_url ON pages(url)')
        cursor.execute('CREATE INDEX IF NOT EXISTS idx_links_source ON links(source_url)')

        self.conn.commit()

    def insert_page(self, url, domain, title, abstract, authors, content, is_article, depth, status_code, crawled_at):
        cursor = self.conn.cursor()
        cursor.execute('''
            INSERT OR IGNORE INTO pages (url, domain, title, abstract, authors, content, is_article, depth, status_code, crawled_at)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        ''', (url, domain, title, abstract, authors, content, is_article, depth, status_code, crawled_at))
        self.conn.commit()

    def insert_links_batch(self, link_tuples):
        if not link_tuples:
            return
        cursor = self.conn.cursor()
        cursor.executemany('''
            INSERT INTO links (source_url, target_url)
            VALUES (?, ?)
        ''', link_tuples)
        self.conn.commit()

    def close(self):
        self.conn.close()

db = Database(DB_PATH)
print("Đã kết nối và khởi tạo CSDL với cấu trúc chuẩn.")

Đã kết nối và khởi tạo CSDL với cấu trúc chuẩn.


In [20]:
# Task 6: Các hàm bổ trợ lọc URL & Trích xuất dữ liệu
IGNORED_EXTENSIONS = (
    '.jpg', '.jpeg', '.png', '.gif', '.svg', '.webp', '.ico',
    '.css', '.js', '.zip', '.tar', '.gz', '.pdf', '.mp4', '.mp3'
)

# Pattern nhận diện bài báo Nature (/articles/s...)
ARTICLE_URL_REGEX = r"^https://www\.nature\.com/articles/[a-zA-Z0-9\.-]+$"

def is_article_url(url: str) -> bool:
    """Kiểm tra URL có phải bài báo khoa học của Nature hay không."""
    return bool(re.match(ARTICLE_URL_REGEX, url))

def is_allowed_navigation_url(url: str) -> bool:
    """Lọc các đường dẫn chuyển hướng tránh lặp hoặc truy cập trang không chứa bài báo."""
    parsed = urlparse(url)
    path = parsed.path.lower()
    query = parsed.query.lower()

    if parsed.netloc.lower() not in ALLOWED_DOMAINS:
        return False

    # Chặn các tham số tìm kiếm/phân trang sâu gây lặp vô tận
    blocked_params = ["order=", "page=", "search=", "filter="]
    if any(param in query for param in blocked_params):
        return False

    # Chặn các trang tài khoản, tìm kiếm, dịch vụ
    blacklists = [
        "/search", "/siteindex", "/user", "/register",
        "/login", "/subscribe", "/authors", "/publishing-options",
        "/nature-portfolio", "/subjects"
    ]
    if any(black in path for black in blacklists):
        return False

    return True

def extract_and_filter_links(soup, base_url, allowed_domains):
    valid_links = []

    for tag in soup.find_all("a", href=True):
        href = tag.get("href", "").strip()
        if href.startswith(("mailto:", "javascript:", "tel:", "#")):
            continue

        full_url = urljoin(base_url, href)
        parsed = urlparse(full_url)

        if parsed.scheme not in ("http", "https"):
            continue

        if parsed.netloc.lower() not in allowed_domains:
            continue

        if any(parsed.path.lower().endswith(ext) for ext in IGNORED_EXTENSIONS):
            continue

        clean_url = f"{parsed.scheme}://{parsed.netloc}{parsed.path}"
        if parsed.query:
            clean_url += f"?{parsed.query}"
        clean_url = clean_url.rstrip("/")

        if is_article_url(clean_url) or is_allowed_navigation_url(clean_url):
            valid_links.append(clean_url)

    return list(set(valid_links))

def extract_page_data(html_content: str) -> tuple[bool, dict, BeautifulSoup]:
    soup = BeautifulSoup(html_content, "html.parser")

    # Kiểm tra bài báo bằng meta tag DOI hoặc Highwire Press
    doi_tag = soup.find("meta", {"name": "citation_doi"}) or soup.find("meta", {"name": "dc.identifier"})

    if not doi_tag or not doi_tag.get("content"):
        # Nếu là trang danh mục/điều hướng
        title = soup.title.get_text(strip=True) if soup.title else "Navigation Page"
        soup_clean = BeautifulSoup(html_content, "html.parser")
        for tag in soup_clean(["script", "style", "noscript", "nav", "footer", "aside"]):
            tag.decompose()
        content = soup_clean.get_text(separator=" ", strip=True)
        return False, {
            "title": title,
            "abstract": "",
            "authors": "",
            "content": content,
            "is_article": 0
        }, soup

    # Trích xuất thông tin bài báo chuẩn
    title_tag = soup.find("meta", {"name": "citation_title"}) or soup.find("meta", {"property": "og:title"})
    title = title_tag["content"].strip() if title_tag and title_tag.get("content") else (
        soup.title.get_text(strip=True) if soup.title else "No Title"
    )

    abstract_tag = soup.find("meta", {"name": "citation_abstract"}) or soup.find("meta", {"name": "description"})
    abstract = abstract_tag["content"].strip() if abstract_tag and abstract_tag.get("content") else ""

    authors_tags = soup.find_all("meta", {"name": "citation_author"})
    authors_list = [tag["content"].strip() for tag in authors_tags if tag.get("content")]
    authors_str = ", ".join(authors_list)

    soup_clean = BeautifulSoup(html_content, "html.parser")
    for tag in soup_clean(["script", "style", "noscript", "nav", "footer", "aside"]):
        tag.decompose()
    content = soup_clean.get_text(separator=" ", strip=True)

    return True, {
        "title": title,
        "abstract": abstract,
        "authors": authors_str,
        "content": content,
        "is_article": 1
    }, soup

print("Đã sẵn sàng các hàm trích xuất dữ liệu.")

Đã sẵn sàng các hàm trích xuất dữ liệu.


In [21]:
# Task 7: Vòng lặp Crawler chính (Main Crawler Loop)
print("=== BẮT ĐẦU CRAWL NATURE ===")
frontier.add_url(SEED_URL, depth=0)
page_count = 0

while not frontier.is_empty() and page_count < MAX_PAGES:
    current_url, depth = frontier.get_next()

    try:
        start_time = time.time()
        response = requests.get(current_url, headers=HEADERS, timeout=REQUEST_TIMEOUT)

        if response.status_code != 200:
            continue

        crawled_at = datetime.now().isoformat()
        domain = urlparse(current_url).netloc

        is_article, data, soup = extract_page_data(response.text)

        # Tránh trùng lặp nội dung
        if detector.is_duplicate(data["content"]):
            continue

        # Trích xuất danh sách liên kết
        extracted_links = extract_and_filter_links(soup, current_url, ALLOWED_DOMAINS)
        link_tuples = []

        for link_url in extracted_links:
            if frontier.add_url(link_url, depth + 1):
                link_tuples.append((current_url, link_url))

        # Thêm dữ liệu vào SQLite đúng theo tham số cấu trúc cũ
        db.insert_page(
            url=current_url,
            domain=domain,
            title=data["title"],
            abstract=data["abstract"],
            authors=data["authors"],
            content=data["content"],
            is_article=data["is_article"],
            depth=depth,
            status_code=response.status_code,
            crawled_at=crawled_at
        )
        db.insert_links_batch(link_tuples)

        elapsed_time = time.time() - start_time

        if is_article:
            page_count += 1
            print(f"[{page_count:03d}] [ARTICLE] Depth: {depth} | Links: {len(extracted_links)} | Time: {elapsed_time:.2f}s | {data['title'][:40]}... | URL: {current_url}")
        else:
            print(f"[NAV PAGE] Depth: {depth} | Links: {len(extracted_links)} | Navigating: {current_url}")

        time.sleep(CRAWL_DELAY)

    except Exception as e:
        print(f"Lỗi khi cào URL {current_url}: {e}")

db.close()
print("=== HOÀN THÀNH CRAWL NATURE ===")

=== BẮT ĐẦU CRAWL NATURE ===
=== HOÀN THÀNH CRAWL NATURE ===
